# 15 — Text as a Sequence: Preparing Sentences for an RNN

**Learning objective:** See exactly how a natural-language sentence becomes a tensor sequence that a recurrent network can process.

This notebook extends the RNN track from numeric time series to **token sequences**:

```text
raw sentence
   ↓
tokens
   ↓
vocabulary IDs
   ↓
fixed-length padded sequence
   ↓
embedding vector at each timestep
   ↓
RNN hidden states
   ↓
sentence representation
```

The important idea is that RNNs do not read words directly. They receive **vectors over ordered timesteps**.

## 1. The text data contract

Our local dataset contains one sentence, one binary sentiment label, and the source domain.

- `label = 1`: positive sentiment
- `label = 0`: negative sentiment
- domains: Amazon, IMDb, Yelp

For an NLP RNN, **time** is token position rather than clock time.

In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED=42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

DATA=Path("RNN/data/sentence_sentiment_uci.csv")
df=pd.read_csv(DATA)
print("TensorFlow:",tf.__version__)
print("shape:",df.shape)
print(df.groupby(["source","label"]).size().unstack())
display(df.sample(6,random_state=SEED))

2026-09-21 11:27:11.305000: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-21 11:27:11.307576: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-21 11:27:11.315791: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789990031.329601    2513 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789990031.333714    2513 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-21 11:27:11.348106: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

TensorFlow: 2.18.1
shape: (3000, 3)
label     0    1
source          
amazon  500  500
imdb    500  500
yelp    500  500


,source,text,label
1801,imdb,Avoid at ALL costs!,0
1190,imdb,"Garbo, who showed right off the bat that her t...",1
1817,imdb,You will leave the theater wanting to go out a...,1
251,amazon,O my gosh the best phone I have ever had.,1
2505,yelp,I would not recommend this place.,0
1117,imdb,Then scene where they debated whether or not t...,0


## 2. Tokenization and integer IDs

`TextVectorization` learns a vocabulary. Each token becomes an integer index.

Changing the vocabulary size changes what is represented explicitly versus mapped to `[UNK]`.

Changing `output_sequence_length` changes how much sentence context is preserved before truncation and how much padding is added to shorter sentences.

In [2]:
train_text=df.sample(frac=.70,random_state=SEED)["text"].astype(str).to_numpy()

vectorizer=tf.keras.layers.TextVectorization(
    max_tokens=5000,
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    output_mode="int",
    output_sequence_length=20,
)
vectorizer.adapt(train_text)

vocab=vectorizer.get_vocabulary()
print("vocabulary size:",len(vocab))
print("special tokens + first learned tokens:",vocab[:20])

examples=tf.constant([
    "The movie was excellent and beautifully acted.",
    "The battery was terrible and stopped working."
])
ids=vectorizer(examples)
print("integer tensor shape:",ids.shape)
print(ids.numpy())

vocabulary size: 4373
special tokens + first learned tokens: ['', '[UNK]', np.str_('the'), np.str_('and'), np.str_('i'), np.str_('a'), np.str_('is'), np.str_('to'), np.str_('it'), np.str_('this'), np.str_('of'), np.str_('was'), np.str_('in'), np.str_('not'), np.str_('for'), np.str_('that'), np.str_('with'), np.str_('my'), np.str_('very'), np.str_('good')]
integer tensor shape: (2, 20)
[[   2   27   11  109    3 1647 1691    0    0    0    0    0    0    0
     0    0    0    0    0    0]
 [   2  115   11  159    3  855  380    0    0    0    0    0    0    0
     0    0    0    0    0    0]]


2026-09-21 11:27:12.954657: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
id_to_token=dict(enumerate(vocab))
for sentence,row in zip(examples.numpy(),ids.numpy()):
    decoded=[id_to_token.get(int(i),"?") for i in row if i!=0]
    print("\nraw:",sentence.decode())
    print("decoded vectorized sequence:",decoded)


raw: The movie was excellent and beautifully acted.
decoded vectorized sequence: [np.str_('the'), np.str_('movie'), np.str_('was'), np.str_('excellent'), np.str_('and'), np.str_('beautifully'), np.str_('acted')]

raw: The battery was terrible and stopped working.
decoded vectorized sequence: [np.str_('the'), np.str_('battery'), np.str_('was'), np.str_('terrible'), np.str_('and'), np.str_('stopped'), np.str_('working')]


## 3. Padding and masking

A batch needs rectangular tensors, but sentences have different lengths.

Padding uses ID `0`. With `Embedding(mask_zero=True)`, TensorFlow marks padded timesteps so compatible recurrent layers do not treat padding as meaningful language.

This is the text equivalent of saying: **these timesteps do not contain observations**.

In [4]:
embed=tf.keras.layers.Embedding(
    input_dim=len(vocab),
    output_dim=8,
    mask_zero=True
)
embedded=embed(ids)
mask=embed.compute_mask(ids)

print("token IDs:",ids.shape)
print("embedded sequence:",embedded.shape)
print("mask:",mask.shape)
print(mask.numpy().astype(int))

token IDs: (2, 20)
embedded sequence: (2, 20, 8)
mask: (2, 20)
[[1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0]]


## 4. Hidden state over tokens

For sentence classification we usually want one vector representing the whole sentence.

With `return_sequences=True`, a SimpleRNN exposes one hidden vector for every token position. The final valid hidden state can act as the sentence representation.

In [5]:
rnn=tf.keras.layers.SimpleRNN(6,return_sequences=True,return_state=True)
all_states,final_state=rnn(embedded,mask=mask)

print("all hidden states:",all_states.shape)
print("final sentence state:",final_state.shape)
print("first sentence final representation:")
print(np.round(final_state.numpy()[0],3))

all hidden states: (2, 20, 6)
final sentence state: (2, 6)
first sentence final representation:
[ 0.042 -0.096 -0.055  0.011 -0.018 -0.071]


## Change map

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Increase sequence length | more tokens retained | more context, more recurrent steps, more compute |
| Reduce vocabulary | more tokens become `[UNK]` | information loss rises |
| Increase embedding dimension | richer token vectors | more parameters and capacity |
| Increase hidden units | larger sentence state | more memory/capacity and overfitting risk |
| Enable masking | padding is ignored | state represents real tokens rather than zeros |

## What you should now be able to explain

A sentence classifier is not magic:

`sentence → token IDs → embeddings → hidden-state updates → final sentence vector → classifier`.

The next notebook trains that complete pipeline in TensorFlow.